# Characteristic Mode Analysis — Plaque rectangulaire

Extraction des **5 premiers modes caracteristiques** d'une plaque conductrice rectangulaire  
L x W = 0.3 m x 0.2 m a f = 500 MHz (lambda = 0.6 m).  
La frequence correspond a la **premiere resonance TM10** (L = lambda/2).

**Probleme aux valeurs propres generalise (Harrington-Mautz) :**
$$X J_n = \lambda_n R J_n, \qquad Z = R + jX$$

| lambda_n | Interpretation |
|:---:|:---|
| approx 0 | Mode **resonant** - rayonne efficacement |
| > 0 | Mode **inductif** - stocke de l'energie magnetique |
| < 0 | Mode **capacitif** - stocke de l'energie electrique |

In [1]:
import numpy as np
import time

from backend.utils.gmsh_function import *
from backend.utils.file_path import setup_save_file_paths
from backend.rwg.rwg1 import (
    Points, Triangles, Edges,
    DataManager_rwg1, load_mesh_file, filter_complexes_jonctions
)
from backend.rwg.rwg2 import Barycentric_triangle, Vecteurs_Rho, DataManager_rwg2
from backend.rwg.rwg3 import calculate_z_matrice, DataManager_rwg3
from backend.cma.cma_solver import CharacteristicModeSolver
from backend.cma.cma_parameters import CMAParameters

In [2]:
name = "plate_cma"
path = setup_save_file_paths(name)

mm = 1e-3
length = 300 * mm
width  = 200 * mm

light_speed = 3e8
frequency   = 500e6
wavelength  = light_speed / frequency
mesh_size   = wavelength / 25

print(f"Frequence : {frequency/1e6:.0f} MHz  |  lambda = {wavelength:.3f} m")
print(f"Plaque    : L = {length:.3f} m = {length/wavelength:.2f}lam  x  W = {width:.3f} m = {width/wavelength:.2f}lam")
print(f"Maillage  : mesh_size = lambda/{int(round(wavelength/mesh_size))} = {mesh_size*100:.1f} cm")

Frequence : 500 MHz  |  lambda = 0.600 m
Plaque    : L = 0.300 m = 0.50lam  x  W = 0.200 m = 0.33lam
Maillage  : mesh_size = lambda/25 = 2.4 cm


## 1. Generation du maillage Gmsh

In [3]:
gmsh.initialize()
gmsh.model.add("plate_cma")
setup_performance_config()

gmsh.model.occ.addRectangle(-length / 2, -width / 2, 0, length, width)
gmsh.model.occ.synchronize()

generate_and_save_mesh(path, mesh_size)

# uncomment to visualize the mesh
#gmsh.fltk.run()

gmsh.finalize()

[PERFORMANCE] Gmsh configured to utilize 16 threads.
Geometry file saved in data/gmsh_files/plate_cma.brep successfully
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
Mesh file saved in data/gmsh_files/plate_cma.msh successfully


## 2. Pipeline MoM : rwg1 -> rwg2 -> rwg3

La CMA ne necessite **aucune source d'excitation** : seule la matrice Z est calculee.

In [4]:
# rwg1 : chargement et pretraitement du maillage
extract_msh_to_mat(path.msh, path.mat)
p, t = load_mesh_file(path.mat)

points    = Points(p)
triangles = Triangles(t)
triangles.calculate_triangles_area_and_center(points)
edges = triangles.get_edges()
filter_complexes_jonctions(points, triangles, edges)
edges.compute_edges_length(points)
DataManager_rwg1.save_data(path, points, triangles, edges)

N = edges.total_number_of_edges
print(f"Maillage : {points.total_of_points} sommets  {triangles.total_of_triangles} triangles  {N} aretes RWG (DOF={N})")

# rwg2 : centres barycentriques et vecteurs rho
bary = Barycentric_triangle()
bary.calculate_barycentric_center(points, triangles)
rho = Vecteurs_Rho()
rho.calculate_vecteurs_rho(points, triangles, edges, bary)
DataManager_rwg2.save_data(path, bary, rho)

# rwg3 : matrice d'impedance Z (sans excitation)
t0 = time.time()
omega, mu, epsilon, c_light, eta, matrice_z = calculate_z_matrice(
    triangles, edges, bary, rho, frequency
)
DataManager_rwg3.save_data(path, frequency, omega, mu, epsilon, c_light, eta, matrice_z)
print(f"Matrice Z : {matrice_z.shape[0]}x{matrice_z.shape[1]}  calculee en {time.time()-t0:.2f} s")

Maillage : 162 sommets  278 triangles  395 aretes RWG (DOF=395)
Matrice Z : 395x395  calculee en 0.12 s


## 3. Verification des proprietes de Z

In [5]:
R = np.real(matrice_z)
X = np.imag(matrice_z)
eigvals_R = np.linalg.eigvalsh(R)

print("=== Proprietes de Z pour la CMA ===")
print(f"  Erreur symetrie R : {np.max(np.abs(R - R.T)):.2e}")
print(f"  Erreur symetrie X : {np.max(np.abs(X - X.T)):.2e}")
print(f"  lam_min(R)        : {eigvals_R.min():.4e}  (requis >= 0 pour PEC sans pertes)")
print(f"  lam_max(R)        : {eigvals_R.max():.4e}")
print(f"  cond(R)           : {np.linalg.cond(R):.2e}")

=== Proprietes de Z pour la CMA ===
  Erreur symetrie R : 2.96e-06
  Erreur symetrie X : 7.33e-03
  lam_min(R)        : -6.9619e-05  (requis >= 0 pour PEC sans pertes)
  lam_max(R)        : 2.7826e-02
  cond(R)           : 5.14e+19


## 4. Solveur CMA — 5 modes

In [6]:
n_modes = 5
params  = CMAParameters(n_modes=n_modes, regularization_epsilon=1e-10, sort_by='magnitude')
solver  = CharacteristicModeSolver(path, params)

modal_data = solver.solve()

RuntimeError: CMA: eigenvalue decomposition failed. R is not positive definite even after regularization. Try increasing regularization_epsilon.

## 5. Valeurs propres caracteristiques

In [ ]:
print("=" * 52)
print(f"  Mode   {'lambda_n':>18}   {'Type':>12}")
print("-" * 52)
for n, lam in enumerate(modal_data.eigenvalues):
    if abs(lam) < 1.0:
        label = "RESONANT"
    elif lam > 0:
        label = "inductif"
    else:
        label = "capacitif"
    print(f"  {n+1:>4}   {lam:>18.4f}   {label:>12}")
print("=" * 52)

  Mode             lambda_n           Type
----------------------------------------------------


NameError: name 'modal_data' is not defined

## 6. Distribution de courant modale

La densite de courant surfacique |J_n(r)| (A/m) est reconstruite sur chaque triangle :
$$J_n(T) = \sum_{m \in T} J_n[m] \cdot \frac{l_m}{2 A_T} \rho_m(T)$$

In [ ]:
figures = solver.visualize(modal_data, show=False)

NameError: name 'modal_data' is not defined

In [ ]:
figures[0]

NameError: name 'figures' is not defined

In [ ]:
figures[1]

NameError: name 'figures' is not defined

In [ ]:
figures[2]

NameError: name 'figures' is not defined

In [ ]:
figures[3]

NameError: name 'figures' is not defined

In [ ]:
figures[4]

NameError: name 'figures' is not defined